In [21]:
# import libraries
import scMPRAforge as scm 
import pandas as pd
import urllib.request
import subprocess 
import h5py
from scipy.sparse import csc_matrix
import scipy.sparse as sp
import scanpy as sc
from anndata import AnnData
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import sys
#import scikit

In [4]:
# define my working direcotry
data_root="/home/sxl6/project_pi_skr2/sxl6/tabula_data/seelig"

In [5]:
# read the dna file 
dna=pd.read_csv(f"{data_root}/dna.tsv",sep="\t",index_col=0)

# read the mpra file
counts_groupby_cre = pd.read_csv(f"{data_root}/susanna_seelig_counts_grouped.txt", sep='\t')

# add the DNA sequence information 
plus_DNA=pd.merge(counts_groupby_cre,dna,on="cre_id",how="left",validate="many_to_one")

plus_DNA

,cell_bc,rep_id,cre_id,cell_type,umis_mpra_bc,reads_DNA
0,A9_A2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
1,A6_A2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
2,A2_B1_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
3,A5_B2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
4,A1_B2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
...,...,...,...,...,...,...
14310795,A7_F6_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
14310796,A12_F7_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451
14310797,A5_F7_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
14310798,A5_F8_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451


# get the counts for HEPG2 only

In [39]:
# get the HEPG2 cell type only 
HEPG2 = rep1.loc[rep1['cell_type'] == 'HEPG2'].copy()
HEPG2

,cell_bc,rep_id,cre_id,cell_type,umis_mpra_bc,reads_DNA
0,A9_A2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
2,A2_B1_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
4,A1_B2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
6,A4_A1_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
8,A12_B4_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,0,1607
...,...,...,...,...,...,...
14310787,A12_F1_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451
14310788,A11_F1_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451
14310790,A3_F2_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451
14310794,A4_F6_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,HEPG2,0,12451


In [40]:


HEPG2_agg = (
    HEPG2
    .groupby(["cre_id", "cell_type"], as_index=False)
    .agg(
        reads_DNA=("reads_DNA", "sum"),
        reads_RNA=("umis_mpra_bc", "sum")

    ))
HEPG2_agg.head()

HEPG2_agg['log2FCHEPG2'] = np.log2(
    (HEPG2_agg['reads_RNA'] + 1) /
    (HEPG2_agg['reads_DNA'] + 1))

HEPG2_agg



,cre_id,cell_type,reads_DNA,reads_RNA,log2FCHEPG2
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,HEPG2,8578166,24,-18.388382
1,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,HEPG2,145871526,3466,-15.360651
2,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,HEPG2,39164906,34,-20.093775
3,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,HEPG2,68780130,147,-18.826035
4,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,HEPG2,10216932,150,-16.046054
...,...,...,...,...,...
1340,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,HEPG2,53540140,85,-19.247853
1341,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,HEPG2,18389410,9,-20.810444
1342,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,HEPG2,21693632,594,-15.154022
1343,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,HEPG2,21907152,58,-18.502256


In [35]:

K562 = rep1.loc[rep1['cell_type'] == 'K562'].copy()
K562

,cell_bc,rep_id,cre_id,cell_type,umis_mpra_bc,reads_DNA
1,A6_A2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
3,A5_B2_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
5,A7_B3_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
7,A5_A1_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
10,A7_F11_A2,rep1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,0,1607
...,...,...,...,...,...,...
14310793,A6_F5_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
14310795,A7_F6_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
14310797,A5_F7_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451
14310798,A5_F8_F8,rep1,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,K562,0,12451


In [41]:


K562_agg = (
    K562
    .groupby(["cre_id", "cell_type"], as_index=False)
    .agg(
        reads_DNA=("reads_DNA", "sum"),
        reads_RNA=("umis_mpra_bc", "sum")

    ))
K562_agg.head()

K562_agg['log2FCK562'] = np.log2(
    (K562_agg['reads_RNA'] + 1) /
    (K562_agg['reads_DNA'] + 1))

K562_agg



,cre_id,cell_type,reads_DNA,reads_RNA,log2FCK562
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,K562,8520314,26,-18.267588
1,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,K562,144887754,368,-18.582883
2,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,K562,38900774,87,-18.753864
3,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,K562,68316270,39,-20.703798
4,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,K562,10148028,19,-18.952768
...,...,...,...,...,...
1340,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,K562,53179060,65,-19.619961
1341,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,K562,18265390,2,-22.537647
1342,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,K562,21547328,70,-18.211259
1343,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,K562,21759408,43,-18.915704


In [ ]:
merged = (
    HEPG2_agg[["cre_id", "log2FCHEPG2"]]
    .merge(
        K562_agg[["cre_id", "log2FCK562"]],
        on="cre_id",
        how="inner"
    )
)
merged["sum"] = merged["log2FCHEPG2"] + merged["log2FCK562"]



,cre_id,sum
34,AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACT...,-46.544431
405,CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAG...,-43.809629


In [49]:
lowest_two_cre_ids = merged.nsmallest(2, "sum")["cre_id"].tolist()
lowest_two_cre_ids

['AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT',
 'CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT']